In [1]:
import sys
sys.dont_write_bytecode = True

# from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
# PATH = "D://LLM//gemma//gemma3_4b"
PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

# dev

In [ ]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration

def prefill_infer(model: Gemma3ForConditionalGeneration, 
          input_ids: List[torch.Tensor], 
          kv_caches: List[DynamicCache]):
    if len(input_ids) == 0:
        return None, []
    try:
        cache = None

        # ----- 整合所有input ----- #
        _input_ids = torch.cat(input_ids, dim=0)
        cache = cache_manager.KVCache_merge(kv_caches)

        # ----- Prefilling過程 ----- #
        with torch.no_grad():
            model(
                input_ids=torch.LongTensor(_input_ids).to(model.device),
                use_cache=True,
                past_key_values=cache,
                )
        print("Prefill Cache Shape:", cache.key_cache[0].shape)
        # ----- 拆解cache ----- #
        eds = []
        for ids in input_ids:
            _list = torch.where(ids==0)[1].tolist()
            if _list:
                eds += [_list[0] - ids.shape[1]]
            else:
                eds += [None]
        caches = cache_manager.KVCache_split(cache,eds)
    finally:
        for item in ("input_ids", ):
            exec(f"del {item}")
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return None, caches


In [32]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration
MAX_NEW_TOKENS_SIZE = 16

def decode_infer(model: Gemma3ForConditionalGeneration, 
          input_ids: List[torch.Tensor], 
          kv_caches: List[DynamicCache]):
    if len(input_ids) == 0:
        return None, []
    try:
        # ----- 宣告物件 ----- #
        device = model.device
        n_batch = len(kv_caches)
        eos_token_ids = [1, 106] # processor.tokenizer.eos_token_id == 1
        unfinished_sequences = torch.ones(n_batch, dtype=torch.long, device=device)
        generated_ids = [[] for _ in range(n_batch)]
        merged_cache = cache_manager.KVCache_merge(kv_caches)
        print("Decode Cache Shape:", merged_cache.key_cache[0].shape)

        # ----- 將input做合併 ----- #
        input_ids_tensor = torch.cat(input_ids, dim=0).to(device)

        # ----- Decoding Loop ----- #
        for step in range(MAX_NEW_TOKENS_SIZE):

            # ----- 全部都做完了 ----- #
            if unfinished_sequences.max() == 0:
                break # Stop Decoding
            
            # ----- 計算position_ids ----- #
            cache_len = merged_cache.get_seq_length(layer_idx=0)
            position_ids = torch.tensor([[cache_len-1]], device=device).expand(n_batch, -1)

            # ----- 生成tokens ----- #
            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids_tensor,
                    past_key_values=merged_cache,
                    position_ids=position_ids,
                    use_cache=True)
            logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(logits, dim=-1)
            for i in range(n_batch):
                if unfinished_sequences[i]:
                    generated_ids[i].append(next_token[i].item())
            input_ids_tensor = next_token.unsqueeze(1)
            is_eos = torch.isin(next_token, torch.tensor(eos_token_ids, device=device))
            unfinished_sequences.mul_(~is_eos) # in-place更新

        # ----- 找EOS位置 ----- #
        eds = list()
        for i in range(n_batch):
            generated_length = len(generated_ids[i])
            _idx = generated_length - MAX_NEW_TOKENS_SIZE
            eds += [_idx if _idx < 0 else None]

        # ----- Cache更新 ----- #
        new_caches_list = cache_manager.KVCache_split(merged_cache, eds)
 
    finally:
        for item in ("input_ids", "outputs", "logits", "next_token", "token_id"):
            try:
                exec(f"del {item}")
            except:
                pass
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return generated_ids, new_caches_list

# 測試

In [33]:
import uuid
import importlib
# importlib.reload(scheduler)
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    _scheduler = scheduler.RequestManager()
    ids = tokenizer.encode(MSG.format(prompt=sentences))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    print("token長度:", len(ids))
    _scheduler.add_request(request)

    TEXT = ""
    for _ in range(8):
        d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
        text, DCACHES = decode_infer(llm_model, d_input_ids, d_caches)
        _, PCACHES = prefill_infer(llm_model, p_input_ids, p_caches)
        _scheduler.update(PCACHES)
        if text is not None:
            decode_requests[0].input_ids = torch.tensor(text[0][-1:]).unsqueeze(0)
            decode_requests[0].kv_cache = DCACHES[0]
            _scheduler.add_request(decode_requests[0])
            try:
                TEXT += tokenizer.decode(text[0], skip_special_tokens=True)
            except:
                continue
    print(TEXT, "\n", "-"*50)

token長度: 29
torch.Size([1, 1, 16, 256])
torch.Size([1, 1, 32, 256])
torch.Size([1, 1, 44, 256])
Decode Cache Shape: torch.Size([1, 1, 29, 256])
Decode Cache Shape: torch.Size([1, 1, 45, 256])
Decode Cache Shape: torch.Size([1, 1, 61, 256])
Decode Cache Shape: torch.Size([1, 1, 77, 256])
Decode Cache Shape: torch.Size([1, 1, 93, 256])
半導體廠務通常在做以下幾件事：

1.  **安全確保：**
   -   安全是首要的。安全措施包括：
      -   嚴格的防護措施，包括：
         -   使用安全警示燈
         -   使用安全警示警告燈
         -   使用安全警示警告燈
         - 
 --------------------------------------------------
token長度: 50
torch.Size([1, 1, 16, 256])
torch.Size([1, 1, 32, 256])
torch.Size([1, 1, 48, 256])
torch.Size([1, 1, 64, 256])
torch.Size([1, 1, 65, 256])
Decode Cache Shape: torch.Size([1, 1, 50, 256])
Decode Cache Shape: torch.Size([1, 1, 66, 256])
Decode Cache Shape: torch.Size([1, 1, 82, 256])
Black Scholes' core spirit is to emphasize the importance of **statistical thinking** in making investment decisions. He argues that by understanding the underlyi

In [ ]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt):
    max_seq_len = 16

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                # cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)
                cache_position = torch.arange(past_key_values.get_seq_length(layer_idx=0)-1, 
                                              past_key_values.get_seq_length(layer_idx=0), 
                                              dtype=torch.long, 
                                              device = llm_model.device)
                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                # print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [29]:
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    text1, cache1 = gemma3_resp(sentences)
    print(cache1.key_cache[0].shape)
    print(text1, "\n", "-"*50)

torch.Size([1, 1, 44, 256])
半導體廠務通常在做以下幾個核心任務：

1. 
 --------------------------------------------------
torch.Size([1, 1, 65, 256])
Black Scholes的核心精神是：**“黑色的黑色的黑色的黑 
 --------------------------------------------------
torch.Size([1, 1, 40, 256])
巨單交易是指在一個交易中，**所有的交易者（包括交易 
 --------------------------------------------------
torch.Size([1, 1, 39, 256])
機器學習 (Machine Learning) 是一種人工智能 (Artificial Intelligence) 的方法， 
 --------------------------------------------------
